# 真實 TCCIP GRID：完整資料前處理

本 notebook 是真實資料前處理的 canonical workflow。範圍從原始逐月最高溫 GRID 開始，到 NN-derived GEV parameters 與所有候選空間解釋變數的一對一整合為止。GP kernel selection、Spatial FFS、return level 與最終驗證不在本 notebook 執行。

流程：

```text
TCCIP monthly TMAX files
        -> quality control and annual block maxima
        -> median/IQR and 11 empirical quantiles
        -> pretrained NN and GEV parameters
        -> projected coordinates and spatial predictors
        -> raw isotropy/stationarity diagnostics
        -> model-ready GRID table
```

## 0. 分析設定

預設直接載入已重建的資料，避免每次開啟 notebook 都重新讀取 65 年原始 CSV 與執行 NN。若原始資料或模型權重改變，再將 `RUN_FULL_PREPROCESSING` 設為 `True`。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from data_preprocessing_pipeline import (
    ANALYSIS_START,
    MIN_ANNUAL_OBSERVATIONS,
    MIN_MONTH_COVERAGE,
    RAW_TMAX_DIR,
    load_existing_temperature_tables,
    run_preprocessing_pipeline,
)
from project_paths import (
    FIGURE_DIR,
    PROCESSED_DATA_DIR,
    SPATIAL_PREDICTOR_PROCESSED_DIR,
)
from spatial_diagnostics import plot_isotropy_stationarity_diagnostics

RUN_FULL_PREPROCESSING = False
print('repository:', ROOT)
print('raw TMAX directory:', RAW_TMAX_DIR)

# Part I：原始格點處理

## 1. 原始資料來源與分析單位

原始資料為 TCCIP 網格化觀測月資料的最高溫產品。每年一個 CSV，每個 GRID 具有 12 個月份。`monthly_tmax` 代表該月的最高溫資料值，不是月平均溫度。

每個座標建立固定 GRID identifier：

$$
\operatorname{GRIDID}=G(\operatorname{longitude})\_ (\operatorname{latitude}).
$$

In [ ]:
source_files = sorted(RAW_TMAX_DIR.glob('觀測_月資料_臺灣_最高溫_*.csv'))
source_audit = pd.DataFrame({
    'item': ['number_of_files', 'first_file', 'last_file'],
    'value': [
        len(source_files),
        source_files[0].name if source_files else None,
        source_files[-1].name if source_files else None,
    ],
})
display(source_audit)

## 2. 缺值、時間範圍與 coverage

小於或等於 -90 的 sentinel values 先轉成缺值。分析從 1980 年開始，並保留有效月份比例至少 80% 的 GRID：

$$
C_i=\frac{\text{GRID }i\text{ 的有效月份數}}{\text{1980 年後應有月份數}},
$$

$$
\text{保留 GRID }i\quad\Longleftrightarrow\quad C_i\geq0.80.
$$

In [ ]:
if RUN_FULL_PREPROCESSING:
    outputs = run_preprocessing_pipeline(rebuild_temperature=True)
    temperature = outputs
else:
    temperature = load_existing_temperature_tables()

coverage = temperature['coverage']
display(temperature['summary'])

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.hist(coverage['valid_month_ratio'], bins=30, color='#2b8cbe', edgecolor='white')
ax.axvline(MIN_MONTH_COVERAGE, color='#d95f0e', linestyle='--', label='80% threshold')
ax.set(title='Monthly TMAX coverage after 1980', xlabel='valid month ratio', ylabel='GRID count')
ax.legend()
fig.tight_layout()

## 3. 建立年最大值 block maxima

每個 GRID、每一年由 12 個逐月最高溫取最大值：

$$
M_{i,y}=\max_{m=1,\ldots,12}X_{i,y,m}.
$$

1980 至 2024 年理論上提供 45 個 annual block maxima。每個 GRID 至少需要 30 個有效年份，才能進入 NN 參數估計。

In [ ]:
annual_max = temperature['annual_max']
annual_locations = temperature['annual_locations']
annual_counts = annual_max.notna().sum()
display(annual_counts.describe().to_frame('valid annual maxima'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hist(annual_counts, bins=np.arange(29.5, 46.5, 1), color='#41ab5d', edgecolor='white')
axes[0].axvline(MIN_ANNUAL_OBSERVATIONS, color='crimson', linestyle='--')
axes[0].set(title='Valid annual maxima per GRID', xlabel='number of years', ylabel='GRID count')
annual_mean = annual_max.mean(axis=1)
axes[1].plot(annual_mean.index, annual_mean.values, color='#225ea8')
axes[1].set(title='Taiwan-wide mean annual maximum', xlabel='year', ylabel='temperature')
fig.tight_layout()

## 4. 觀測極端溫度曲面的方向性與 stationarity

先以每個 GRID 的多年平均 annual maximum 建立觀測極端溫度曲面。左圖比較 E--W、NE--SW、N--S 與 NW--SE directional variograms，用於檢查 isotropy；右圖比較 south、central、north 三個 projected-northing bands 的 regional variograms，用於檢查 covariance stationarity。

$$
\gamma_\theta(h)=\frac{1}{2N_\theta(h)}
\sum_{(i,j)\in N_\theta(h)}\{Z(s_i)-Z(s_j)\}^2.
$$

這些圖是候選結構診斷，不是正式拒絕 isotropy 或 stationarity 的單一假設檢定。

In [ ]:
observed_surface = annual_locations.merge(
    annual_max.mean(axis=0).rename_axis('station').rename('mean_annual_max').reset_index(),
    on='station', how='inner', validate='one_to_one',
)
observed_diagnostics, observed_figure = plot_isotropy_stationarity_diagnostics(
    observed_surface,
    value_columns={'annual_max_mean': 'mean_annual_max'},
    value_kind='observed mean annual-maximum temperature surface',
    output_figure_path=FIGURE_DIR / 'preprocessing_observed_grid_isotropy_stationarity.png',
    output_table_path=PROCESSED_DATA_DIR / 'observed_grid_isotropy_stationarity_summary.csv',
)
display(observed_diagnostics)
display(observed_figure)

# Part II：GEV 參數處理

## 5. Median／IQR 與 11 empirical quantiles

每個 GRID 的完整 annual-maximum sample 先做 robust standardization：

$$
z_{i,y}=\frac{M_{i,y}-\operatorname{median}(M_i)}{\operatorname{IQR}(M_i)}.
$$

再由整筆標準化 sample 計算 11 個 empirical quantiles，形成一次 NN input。不是每一年各自產生一組 11 quantiles。

$$
\mathbf q_i=\bigl(Q_i(0.0001),Q_i(0.001),\ldots,Q_i(0.9999)\bigr).
$$

## 6. NN inference 與 GEV shape convention

預訓練 NN 輸出 standardized location、delta 與 SciPy shape `c`。反標準化後保存 location 與 scale，並同時保留兩種 shape 欄位：

$$
\operatorname{shape\_c\_hat}=c,
$$

$$
\operatorname{xi\_hat}=-c.
$$

這個符號轉換不可省略。舊 preprocessing 曾把 `c` 直接標成 `xi_hat`；canonical pipeline 已修正並保留兩欄供稽核。

In [ ]:
if RUN_FULL_PREPROCESSING:
    parameters = outputs['parameters']
    quantiles = outputs['quantiles']
else:
    parameters = pd.read_csv(PROCESSED_DATA_DIR / 'grid_station_gev_params_with_loc.csv')
    quantiles = pd.read_csv(PROCESSED_DATA_DIR / 'real_grid_11_quantiles.csv')

assert np.allclose(parameters['xi_hat'], -parameters['shape_c_hat'])
display(parameters[['n_obs', 'mu_hat', 'sigma_hat', 'log_sigma_hat', 'shape_c_hat', 'xi_hat']].describe().T)
display(quantiles.head())

## 7. NN-derived parameter surfaces

這三張圖是每個真實 GRID 經 NN 推論後的參數估計，不是 GP 平滑或外推結果。

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for axis, column, title in zip(
    axes,
    ['mu_hat', 'log_sigma_hat', 'xi_hat'],
    ['mu', 'log sigma', 'xi'],
):
    points = axis.scatter(parameters['x_km'], parameters['y_km'], c=parameters[column], s=12, cmap='turbo')
    figure.colorbar(points, ax=axis, label=column)
    axis.set(title=title, xlabel='TWD97 Easting (km)', ylabel='TWD97 Northing (km)', aspect='equal')
figure.suptitle('NN-derived GEV parameter surfaces')
figure.tight_layout()

# Part III：空間候選解釋變數

## 8. 投影座標與地形

經緯度只作資料鍵值與顯示。所有距離、K-means、buffer 與 GP length scale 使用 TWD97 / TM2 zone 121 的公里座標。地形候選包括 elevation、slope、northness、eastness、local relief 與 terrain ruggedness。

坡向以兩個連續分量表達：

$$
\operatorname{northness}=\cos(\operatorname{aspect}),
$$

$$
\operatorname{eastness}=\sin(\operatorname{aspect}).
$$

## 9. 2000 年土地覆蓋連續比例

每個 0.05 度 GRID 內計算 ESA CCI 300 m pixels 的面積加權比例。建模保留 urban、forest、agriculture 與 water 的連續比例，不使用 0.5 threshold 分成單一類別；`other_ratio` 作為 composition reference。

$$
p_{g,c}=\frac{\sum_{j\in g}A_j\operatorname{I}(L_j=c)}{\sum_{j\in g}A_j}.
$$

## 10. 海岸距離

使用 GSHHG intermediate-resolution level-1 land/ocean boundary。GRID 與 coastline 投影至 EPSG:3826 後，計算 GRID 中心至最近海岸線的平面距離：

$$
d_{\mathrm{coast}}(s_i)=\min_{u\in\mathcal C}\lVert s_i-u\rVert_2.
$$

`coast_distance_km` 與 `water_ratio` 不相同：前者表示距海遠近，後者表示 GRID 內水域面積比例。

In [ ]:
terrain = pd.read_csv(SPATIAL_PREDICTOR_PROCESSED_DIR / 'tccip_grid_terrain_predictors.csv')
land_cover = pd.read_csv(SPATIAL_PREDICTOR_PROCESSED_DIR / 'tccip_grid_land_cover_2000.csv')
coast = pd.read_csv(SPATIAL_PREDICTOR_PROCESSED_DIR / 'tccip_grid_coast_distance.csv')
model_ready = pd.read_csv(PROCESSED_DATA_DIR / 'model_ready_grid_parameters.csv')

predictor_maps = [
    ('elevation_m', 'Elevation (m)'),
    ('local_relief_m', 'Local relief (m)'),
    ('agriculture_ratio', 'Agriculture ratio'),
    ('forest_ratio', 'Forest ratio'),
    ('urban_ratio', 'Urban ratio'),
    ('coast_distance_km', 'Coast distance (km)'),
]
figure, axes = plt.subplots(2, 3, figsize=(14, 9))
for axis, (column, title) in zip(axes.ravel(), predictor_maps):
    points = axis.scatter(model_ready['x_km'], model_ready['y_km'], c=model_ready[column], s=10, cmap='viridis')
    figure.colorbar(points, ax=axis, label=column)
    axis.set(title=title, xlabel='Easting (km)', ylabel='Northing (km)', aspect='equal')
figure.suptitle('Candidate spatial predictors aligned to the GEV GRID')
figure.tight_layout()

# Part IV：原始參數曲面的 spatial diagnostics

## 11. Directional 與 regional variograms

對尚未擬合 GP 的 NN-derived parameter surfaces 重複 isotropy／stationarity 診斷。若不同方向曲線明顯不同，將 anisotropic covariance 列為候選；若 south／central／north regional variograms 或區域 mean／variance 差異明顯，將 nonstationary mean、variance 或 range 列為候選。

這一步只建立 candidate plan，不直接決定最終 covariance。

In [ ]:
raw_diagnostics, raw_figure = plot_isotropy_stationarity_diagnostics(
    model_ready,
    value_columns={'mu': 'mu_hat', 'log_sigma': 'log_sigma_hat', 'xi': 'xi_hat'},
    value_kind='raw NN-derived parameter surfaces',
    output_figure_path=FIGURE_DIR / 'preprocessing_raw_parameter_isotropy_stationarity.png',
    output_table_path=PROCESSED_DATA_DIR / 'raw_parameter_isotropy_stationarity_summary.csv',
)
display(raw_diagnostics)
display(raw_figure)

## 12. Model-ready output 與下一步

最終 `model_ready_grid_parameters.csv` 每列是一個真實 GRID，包含正確符號的 GEV parameters、公里座標、地形、2000 年土地覆蓋連續比例與海岸距離。

後續順序為：

```text
raw diagnostics
    -> spatial forward feature selection
    -> kernel / covariance comparison
    -> buffered out-of-fold predictions
    -> OOF residual isotropy/stationarity
    -> mixed RL50 and RL100 pipeline
    -> repeated or nested buffered spatial CV
```

In [ ]:
required = [
    'station', 'mu_hat', 'sigma_hat', 'log_sigma_hat', 'shape_c_hat', 'xi_hat',
    'x_km', 'y_km', 'elevation_m', 'slope_deg', 'northness', 'eastness',
    'local_relief_m', 'terrain_ruggedness_m', 'urban_ratio', 'forest_ratio',
    'agriculture_ratio', 'water_ratio', 'coast_distance_km',
]
assert model_ready[required].isna().sum().sum() == 0
assert model_ready['station'].is_unique
assert np.allclose(model_ready['xi_hat'], -model_ready['shape_c_hat'])
print('model-ready GRID rows:', len(model_ready))
display(model_ready[required].head())